# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret the problem (assume Boston–New York distance if not given) and set up relative motion equation.
Todo #2: Estimate a reasonable Boston–New York distance and justify it briefly.
Todo #3: Compute each train's head start and relative speed, then solve for meeting time and clock time.
Todo #4: Sanity-check the result and present the final meeting time clearly.

Let D be the distance between Boston and New York along the track. Train A leaves Boston at 2:00 pm at 60 mph. 
Train B leaves New York at 3:00 pm at 80 mph, heading toward Boston (so they move toward each other).

Measure time t in hours after 3:00 pm. By 3:00 pm, Train A has a 1-hour head start and has gone 60 miles. After 
3:00 pm, the closing speed is 60 + 80 = 140 mph. They meet when:

60 + 140t = D.

Todo #1: Interpret the problem (assume Boston–New York distance if not given) and set up relative motion equation.
Todo #2: Estimate a reasonable Boston–New York distance and justify it briefly.
Todo #3: Compute each train's head start and relative speed, then solve for meeting time and clock time.
Todo #4: Sanity-check the result and present the final meeting time clearly.

The problem omits D. A common real-world estimate for BostonNew York is about 215 miles (roughly the rail/highway 
corridor distance). Use D \approx 215 mi as a reasonable estimate.

Todo #1: Interpret the problem (assume Boston–New York distance if not given) and set up relative motion equation.
Todo #2: Estimate a reasonable Boston–New York distance and justify it briefly.
Todo #3: Compute each train's head start and relative speed, then solve for meeting time and clock time.
Todo #4: Sanity-check the result and present the final meeting time clearly.

Solve 60 + 140t = 215:

140t = 155 d t = 155/140 d 1.1071 h.

1.1071 h = 1 h + 0.1071 h, and 0.1071 h d 0.1071d60 d 6.43 min.

So they meet about 1 hour 6 minutes after 3:00 pm, i.e. around 4:06 pm.

Todo #1: Interpret the problem (assume Boston–New York distance if not given) and set up relative motion equation.
Todo #2: Estimate a reasonable Boston–New York distance and justify it briefly.
Todo #3: Compute each train's head start and relative speed, then solve for meeting time and clock time.
Todo #4: Sanity-check the result and present the final meeting time clearly.

Check: By ~4:06 pm, Train A has traveled from 2:00 pm for 2.107 h at 60 mph d 126.4 mi. Train B has traveled from 
3:00 pm for 1.107 h at 80 mph d 88.6 mi. Sum d 215 mi, matching the assumed distance. Result is consistent.

Todo #1: Interpret the problem (assume Boston–New York distance if not given) and set up relative motion equation.
Todo #2: Estimate a reasonable Boston–New York distance and justify it briefly.
Todo #3: Compute each train's head start and relative speed, then solve for meeting time and clock time.
Todo #4: Sanity-check the result and present the final meeting time clearly.

Assuming a reasonable Boston–New York distance of about 215 miles:

- By 3:00 pm, the Boston train has a 1-hour head start → it has gone 60 miles.
- After 3:00 pm, they approach each other at a combined speed of 60 + 80 = 140 mph.
- Remaining distance at 3:00 pm: 215 − 60 = 155 miles.
- Time after 3:00 pm to meet:  
  t = 155 / 140 ≈ 1.107 hours ≈ 1 hour 6 minutes.

They meet at approximately 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>